<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<span style="display:inline-block;margin-bottom:10px;padding:2px 12px;border-radius:999px;background:#6366f1;color:#fff;font:600 11px/20px system-ui,sans-serif;letter-spacing:.05em;text-transform:uppercase">Step 9</span>
<h1 style="font-size:1.35em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Step 9: Production</h1>
<p style="margin:.55em 0">Production is configuration, packaging and proof:</p>
<ul style="margin:.4em 0;padding-left:1.4em">
  <li style="margin:.2em 0">a production config (public origin, JSON logs, no demo data, the database on a volume, the key from the environment);</li>
  <li style="margin:.2em 0">one dependency, a non-root container with a health check;</li>
  <li style="margin:.2em 0">a self-check that tests everything without OpenAI, cheap enough for every commit.</li>
</ul>

</div>

In [1]:
%%writefile build/config/production.toml
# Production settings: APP_CONFIG=config/production.toml. The API key comes only from OPENAI_API_KEY.
# Put a TLS-terminating reverse proxy (nginx, Caddy, a cloud load balancer) in front: browsers need
# https + wss for the microphone, and the proxy must allow websocket upgrades on /ws.

[server]
host = "0.0.0.0"
port = 8000
origins = ["https://desk.example.com"]   # the page's public origin(s)
max_sessions = 50
session_limit_s = 600
drain_s = 30
log_format = "json"
log_level = "INFO"

[openai]
model = "gpt-realtime-mini"
voice = "marin"
transcribe_model = "gpt-4o-mini-transcribe"
max_output_tokens = 1000

[resilience]
stall_s = 5
ping_timeout_s = 5
max_resumes = 2

[clinic]
name = "City Care Clinic"

[store]
path = "/data/app.db"   # a mounted volume
seed_demo = false

[otp]
ttl_s = 300
max_tries = 3
max_sends = 3

[skills]
timeout_s = 10
filler_audio = "assets/filler_marin.pcm"

[agents.front_desk]
prompt = "prompts/front_desk.md"
toolsets = ["identity", "registration", "appointments", "clinic_info"]
vad = { type = "server_vad", silence_duration_ms = 900 }


Overwriting build/config/production.toml


In [2]:
%%writefile build/requirements.txt
# The server (python -m app). Python 3.11+ (tomllib). Notebooks need more: see .venv.
websockets==17.4


Overwriting build/requirements.txt


In [3]:
%%writefile build/Dockerfile
FROM python:3.11-slim

ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1 APP_CONFIG=config/production.toml
WORKDIR /srv

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app app
COPY config config
COPY prompts prompts
COPY assets/filler_marin.pcm assets/filler_marin.pcm

# run as a non-root user; the database lives on a mounted volume at /data (store.path in production.toml)
RUN useradd --create-home --uid 10001 desk && mkdir -p /data && chown desk /data
USER desk
VOLUME ["/data"]
EXPOSE 8000

HEALTHCHECK --interval=30s --timeout=5s \
  CMD python -c "import urllib.request,sys; sys.exit(urllib.request.urlopen('http://127.0.0.1:8000/healthz', timeout=3).status != 200)"

# the key comes only from the environment: docker run -e OPENAI_API_KEY=... -v deskdata:/data -p 8000:8000 desk
CMD ["python", "-m", "app"]


Overwriting build/Dockerfile


In [4]:
%%writefile build/.dockerignore
.venv
.flashback
.wwebjs_cache
.playwright-mcp
data
live
web
course
*.ipynb
**/__pycache__
assets/*.npy


Overwriting build/.dockerignore


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">The self-check</h2>
<p style="margin:.55em 0">Every layer, free: settings, registry, the session flow, stall recovery, and the server end to end against a scripted
fake OpenAI (tool call, SMS, the code kept from the model, metrics, the busy limit).</p>

</div>

In [5]:
%%writefile build/app/selfcheck.py
"""Free checks for the whole backend (no OpenAI, no key, no sound card): python -m app.selfcheck

Each check is a function, so notebooks can run them one at a time. A fake OpenAI (a local websocket server
that speaks the Realtime protocol from a script) stands in for the real one.
"""
import asyncio
import concurrent.futures
import json
import re
import tempfile
import time
import urllib.error
import urllib.request
from datetime import datetime
from pathlib import Path

from websockets.asyncio.client import connect
from websockets.asyncio.server import serve
from websockets.exceptions import ConnectionClosed

from app import realtime, registry, server, settings, store
from app.metrics import METRICS
from app.session import Session
from app.verification import Otp, SimulatedSms

TMP = Path(tempfile.mkdtemp(prefix="app-selfcheck-"))


def _sync(coro):
    """Run a coroutine to completion from sync code, inside or outside a running loop (e.g. a notebook)."""
    with concurrent.futures.ThreadPoolExecutor(1) as ex:
        return ex.submit(asyncio.run, coro).result()


def test_settings(env=None) -> settings.Settings:
    """Settings for tests: a fresh temp database, plus any overrides."""
    base = {"APP__STORE__PATH": str(TMP / f"t{time.monotonic_ns()}.db"), "APP__OPENAI__DEV_KEY_INI": '""'}
    return settings.load(env={**base, **(env or {})})


def check_settings():
    s = test_settings({"APP__SERVER__PORT": "9001", "APP__OPENAI__MODEL": "gpt-x"})
    assert s.server.port == 9001 and s.openai.model == "gpt-x" and s.resilience.stall_s == 5.0
    assert s.agents["front_desk"].vad["silence_duration_ms"] == 900
    for env, msg in [({"APP__SERVER__PROT": "1"}, "unknown key"), ({"APP__SERVER__PORT": '"80"'}, "expected int"),
                     ({"APP__STORE__SEED_DEMO": "1"}, "expected bool"), ({"APP__NOPE__X": "1"}, "unknown section"),
                     ({"APP__AGENTS__X__PROMPT": '"prompts/missing.md"', "APP__AGENTS__X__TOOLSETS": '["orders"]'},
                      "no file")]:
        try:
            test_settings(env)
            raise AssertionError(f"accepted {env}")
        except settings.SettingsError as e:
            assert msg in str(e), (env, e)
    try:
        settings.api_key(s, env={})
        raise AssertionError("no key accepted")
    except settings.SettingsError:
        pass
    assert settings.api_key(s, env={"OPENAI_API_KEY": "k"}) == "k"
    print("settings ok")


Overwriting build/app/selfcheck.py


In [6]:
%%writefile -a build/app/selfcheck.py

async def check_registry():
    from typing import Literal
    sets = registry.discover()
    assert {"identity", "registration", "appointments", "clinic_info", "orders"} <= set(sets)

    def doc(f, d):
        f.__doc__ = d
        return f

    bad = [(doc(lambda x: {}, "D."), "first parameter"),
           (doc(lambda ctx, x: {}, "D."), "type hint")]

    def no_doc(ctx):
        return {}

    def weird(ctx, x: list):
        """D."""

    def ghost(ctx, x: int):
        """D.

        Args:
            y: not a parameter
        """
    for fn, msg in [*bad, (no_doc, "docstring"), (weird, "unsupported type"), (ghost, "unknown argument")]:
        try:
            registry.describe(fn, "t")
            raise AssertionError(f"accepted {fn}")
        except registry.SkillDefinitionError as e:
            assert msg in str(e), e

    def demo(ctx, n: int, kind: Literal["a", "b"], note: str = "") -> dict:
        """Does a thing.

        When: when testing.
        Args:
            n: how many
        """
    sk = registry.describe(demo, "demo")
    assert sk.when == "when testing." and sk.description == "Does a thing."
    assert sk.parameters == {"type": "object", "additionalProperties": False, "required": ["n", "kind"],
                             "properties": {"n": {"type": "integer", "description": "how many"},
                                            "kind": {"type": "string", "enum": ["a", "b"]}, "note": {"type": "string"}}}
    p = sk.parameters
    assert registry.validate({"n": 1, "kind": "a"}, p) is None
    for args, msg in [([], "JSON object"), ({"n": 1}, "missing"), ({"n": 1, "kind": "a", "x": 0}, "unknown"),
                      ({"n": True, "kind": "a"}, "boolean"), ({"n": 1.5, "kind": "a"}, "expected integer"),
                      ({"n": 1, "kind": "c"}, "one of")]:
        assert msg in registry.validate(args, p), args

    class Ctx:
        patient = None

    async def calls():
        guarded = registry.Skill("g", "t", "d", "", {"type": "object", "properties": {}, "required": []},
                                 lambda ctx: {"ok": 1}, True)
        assert (await registry.call(guarded, Ctx(), "{}", 1))["status"] == "needs_verification"
        slow = registry.Skill("s", "t", "d", "", guarded.parameters, lambda ctx: time.sleep(2) or {}, False)
        assert "too long" in (await registry.call(slow, Ctx(), "{}", 0.2))["error"]
        boom = registry.Skill("b", "t", "d", "", guarded.parameters, lambda ctx: 1 / 0, False)
        assert "ZeroDivisionError" in (await registry.call(boom, Ctx(), "{}", 1))["error"]
        notdict = registry.Skill("n", "t", "d", "", guarded.parameters, lambda ctx: [1], False)
        assert "invalid result" in (await registry.call(notdict, Ctx(), "{}", 1))["error"]
        assert "too large" in (await registry.call(notdict, Ctx(), " " * 20_000, 1))["error"]
        assert "not valid JSON" in (await registry.call(notdict, Ctx(), "{", 1))["error"]
    await calls()
    print("registry ok")


async def check_session():
    """The whole front-desk flow through Session.run_tool: the exact path a model's proposal takes."""
    s = test_settings()
    store.init(s.path(s.store.path), seed_demo=True)
    registry.discover()
    events, codes = [], iter(["111111", "222222", "333333", "444444"])
    fixed = datetime.combine(datetime.now().date(), datetime.min.time()).replace(minute=1)
    sms = SimulatedSms(on_send=lambda m: events.append({"type": "sms", **m}))
    ses = Session(s, "front_desk", emit=events.append, now=lambda: fixed, sms=sms,
                  otp=Otp(sms, "City Care Clinic", gen=lambda: next(codes)))
    call = lambda tool, **a: _sync(ses.run_tool(tool, json.dumps(a)))  # noqa: E731
    tue, wed = store.next_weekday("Tuesday"), store.next_weekday("Wednesday")

    assert call("my_appointments")["status"] == "needs_verification"                    # guard, in code
    assert "unknown tool" in call("get_order_status", order_id="A123")["error"]        # not this agent's tool
    assert call("verify_patient", phone="9000000000")["status"] == "not_found"
    out = call("verify_patient", phone="98123 45678")
    assert out["status"] == "code_sent" and "111111" not in json.dumps(out) and "Asha" not in json.dumps(out)
    assert call("confirm_code", code="Please read it")["status"] == "need_digits" and ses.otp.pending["tries"] == 0
    assert call("confirm_code", code="000000")["status"] == "wrong_code"
    assert call("confirm_code", code="1 1 1 1 1 1")["status"] == "choose_patient"
    assert call("choose_patient", name="asha")["status"] == "verified" and ses.patient["id"] == 1
    assert [a["time"] for a in call("my_appointments")["appointments"]] == ["10:00 AM"]
    assert call("cancel_appointment", appointment_id=2)["status"] == "not_found"        # Rohan's, not hers
    assert call("book_appointment", doctor="Dr Rao", date=f"{wed}", time="10:40 AM", reason="fever")["status"] == "booked"
    assert call("cancel_appointment", appointment_id=1)["status"] == "cancelled"
    assert call("start_registration", first_name="Meera", last_name="Iyer", phone="9444123456", sex="Female",
                dob="1995-04-03")["status"] == "code_sent"                              # optional args omitted
    assert call("confirm_code", code="222222")["status"] == "registered" and ses.patient["name"] == "Meera Iyer"
    assert call("confirm_code", code="222222")["status"] == "already_verified"
    assert call("verify_patient", phone="9845012367")["status"] == "code_sent"           # 3rd code this call
    assert call("verify_patient", phone="9845012367")["status"] == "too_many_codes"     # 4th: SMS pumping cap
    desk = [e for e in events if e["type"] == "desk"]
    assert not any(c in json.dumps(e["result"]) for e in desk for c in ("111111", "222222", "333333"))
    assert sum(e["type"] == "sms" for e in events) == 3 and desk[-1]["state"]["code_pending_for"] == "verification"
    assert "Today is" in ses.instructions() and "- book_appointment:" in ses.instructions()
    assert METRICS.get("tool_calls_total", skill="book_appointment", status="booked") >= 1
    print("session ok")


Appending to build/app/selfcheck.py


In [7]:
%%writefile -a build/app/selfcheck.py

async def check_resume():
    """An upstream that goes silent mid-response and stops answering pings is detected, and
    converse_resilient reconnects, replays the turns so far, and the reply completes."""
    conns, replayed = [], []
    delta = realtime.b64(bytes(4800))

    async def fake(ws):
        n = len(conns)
        conns.append(ws)
        try:
            if n == 0:
                for _ in range(3):
                    await ws.recv()
                for e in ({"type": "conversation.item.input_audio_transcription.completed", "transcript": "hello"},
                          {"type": "response.created", "response": {"status": "in_progress"}},
                          {"type": "response.output_audio.delta", "item_id": "i1", "delta": delta}):
                    await ws.send(json.dumps(e))
                ws.transport.pause_reading()  # stop reading: pings go unanswered, like the real dead link
                await asyncio.sleep(8)
                return
            while True:
                m = json.loads(await ws.recv())
                if m["type"] == "conversation.item.create":
                    replayed.append((m["item"]["role"], m["item"]["content"][0]["text"]))
                elif m["type"] == "response.create":
                    for e in ({"type": "response.created", "response": {"status": "in_progress"}},
                              {"type": "response.output_audio_transcript.done", "transcript": "carrying on"},
                              {"type": "response.done", "response": {"status": "completed", "output": []}}):
                        await ws.send(json.dumps(e))
        except ConnectionClosed:
            pass

    async def silence():
        while True:
            yield bytes(4800)
            await asyncio.sleep(0.1)

    done, answered = asyncio.Event(), []
    t = time.monotonic()

    def on_note(_t, ev, d):
        if ev == "model_said":
            answered.append(time.monotonic() - t)
            done.set()

    async with serve(fake, "127.0.0.1", 8768):
        log = await realtime.converse_resilient(
            lambda: connect("ws://127.0.0.1:8768"), silence(), server.BrowserSpeaker(asyncio.Queue()), seconds=30,
            context=lambda: "STATE", on_note=on_note, stop=done, stall_s=1, ping_timeout_s=2)
    events = [e for _, e, _ in log]
    assert events.index("STALL") < events.index("RESUME") and len(conns) == 2, events
    assert replayed[0] == ("user", "hello") and replayed[-1][1].endswith("STATE"), replayed
    assert answered and answered[0] < 8, answered
    print(f"resume ok: stall detected, reconnected, reply {answered[0]:.1f}s after the start")


class FakeOpenAI:
    """A scripted Realtime server: on the caller's first audio it 'hears' a phone number and proposes
    verify_patient; after the tool output it asks for the code. Records what our server sent back."""

    def __init__(self):
        self.tool_outputs = []

    async def handler(self, ws):
        try:
            await ws.send(json.dumps({"type": "session.created"}))
            cfg = json.loads(await ws.recv())
            assert cfg["type"] == "session.update" and cfg["session"]["tools"]
            await ws.send(json.dumps({"type": "session.updated"}))
            heard = False
            async for raw in ws:
                m = json.loads(raw)
                if m["type"] == "input_audio_buffer.append" and not heard:
                    heard = True
                    call = {"type": "function_call", "name": "verify_patient", "call_id": "c1",
                            "arguments": json.dumps({"phone": "9845012367"})}
                    for e in ({"type": "input_audio_buffer.speech_started"},
                              {"type": "conversation.item.input_audio_transcription.completed",
                               "transcript": "My number is 9845012367."},
                              {"type": "response.created", "response": {"status": "in_progress"}},
                              {"type": "response.output_item.added", "response_id": "r1", "item": call},
                              {"type": "response.done", "response": {"status": "completed", "output": [call]}}):
                        await ws.send(json.dumps(e))
                elif m["type"] == "conversation.item.create" and m["item"]["type"] == "function_call_output":
                    self.tool_outputs.append(m["item"]["output"])
                elif m["type"] == "response.create":
                    for e in ({"type": "response.created", "response": {"status": "in_progress"}},
                              {"type": "response.output_audio.delta", "item_id": "a1", "delta": realtime.b64(bytes(9600))},
                              {"type": "response.output_audio_transcript.done", "transcript": "Please read me the code."},
                              {"type": "response.done", "response": {"status": "completed", "output": []}}):
                        await ws.send(json.dumps(e))
        except ConnectionClosed:
            pass


def _get(url):
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            return r.status, r.read().decode()
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode()


async def check_server():
    """The real server, end to end, against the fake OpenAI: page, health, metrics, agent allowlist,
    a conversation with a tool call + SMS, and the concurrency limit."""
    port, up = 8781, 8782
    s = test_settings({"APP__SERVER__PORT": str(port), "APP__SERVER__MAX_SESSIONS": "1",
                       "APP__OPENAI__URL": f'"ws://127.0.0.1:{up}"'})
    fake = FakeOpenAI()
    app = server.App(s, key="test-key-not-real")
    calls0 = METRICS.get("tool_calls_total", skill="verify_patient", status="code_sent")
    rejected0 = METRICS.get("sessions_rejected_total")
    ready, stop = asyncio.Event(), asyncio.Event()
    async with serve(fake.handler, "127.0.0.1", up):
        srv = asyncio.create_task(server.run(s, ready=ready, stop=stop, app=app))
        await asyncio.wait_for(ready.wait(), 10)
        base = f"http://127.0.0.1:{port}"
        status, page = await asyncio.to_thread(_get, base + "/")
        assert status == 200 and "/ws?agent=" in page
        status, body = await asyncio.to_thread(_get, base + "/healthz")
        assert status == 200 and json.loads(body)["status"] == "ok"
        assert (await asyncio.to_thread(_get, base + "/nope"))[0] == 404

        async with connect(f"ws://127.0.0.1:{port}/ws?agent=nope") as ws:
            assert "unknown agent" in json.loads(await ws.recv())["message"]

        msgs, audio = [], 0
        async with connect(f"ws://127.0.0.1:{port}/ws?agent=front_desk", max_size=2 ** 22) as ws:
            while json.loads(await ws.recv()).get("state") != "ready":
                pass
            async with connect(f"ws://127.0.0.1:{port}/ws?agent=front_desk") as second:  # max_sessions = 1
                assert "busy" in json.loads(await second.recv())["message"]
            await ws.send(bytes(4800))
            await ws.send(json.dumps({"played": 0}))
            while True:
                m = await asyncio.wait_for(ws.recv(), 10)
                if isinstance(m, bytes):
                    audio += len(m)
                    continue
                m = json.loads(m)
                msgs.append(m)
                if m.get("event") == "model_said":
                    break
        kinds = [m["type"] for m in msgs]
        desk = next(m for m in msgs if m["type"] == "desk")
        sms = next(m for m in msgs if m["type"] == "sms")
        code = re.sub(r"\D", "", sms["text"])
        assert desk["skill"] == "verify_patient" and desk["result"]["status"] == "code_sent", desk
        assert desk["state"]["code_pending_for"] == "verification" and "log" in kinds
        assert fake.tool_outputs and code not in fake.tool_outputs[0], fake.tool_outputs  # the model never saw it
        assert audio >= len(app.filler or b"") + 9600, audio                                 # filler + the reply
        await asyncio.sleep(0.3)
        status, metrics = await asyncio.to_thread(_get, base + "/metrics")
        assert status == 200 and 'tool_calls_total{skill="verify_patient",status="code_sent"}' in metrics
        assert METRICS.get("tool_calls_total", skill="verify_patient", status="code_sent") == calls0 + 1
        assert METRICS.get("sessions_rejected_total") == rejected0 + 1 and "sessions_active 0" in metrics, metrics
        stop.set()
        await asyncio.wait_for(srv, 15)
    print("server ok: page, health, metrics, agent allowlist, tool call + SMS end to end, busy limit")


Appending to build/app/selfcheck.py


In [8]:
%%writefile -a build/app/selfcheck.py

async def check_all():
    check_settings()
    await check_registry()
    await check_session()
    await check_resume()
    await check_server()
    print("ALL OK")


def main():
    asyncio.run(check_all())


if __name__ == "__main__":
    main()


Appending to build/app/selfcheck.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Prove the rebuilt backend</h2>

</div>

In [9]:
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'app.selfcheck'], cwd='build', capture_output=True, text=True, timeout=300)
print('\n'.join(l for l in r.stdout.splitlines() if l.strip()) or r.stderr[-2000:])

settings ok
registry ok
session ok
resume ok: stall detected, reconnected, reply 4.0s after the start
server ok: page, health, metrics, agent allowlist, tool call + SMS end to end, busy limit
ALL OK


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">And the build is the production code, file for file:</p>

</div>

In [10]:
from pathlib import Path

def lines(p):
    return [l.rstrip() for l in Path(p).read_text(encoding='utf-8').splitlines() if l.strip()]

files = [p.relative_to('build') for p in Path('build').rglob('*') if p.is_file() and p.suffix in ('.py', '.toml', '.md', '.html')
         and 'data' not in p.parts and '__pycache__' not in p.parts]
same = {str(f): lines(Path('build') / f) == lines(Path('..') / f) for f in files}
len(files), [f for f, ok in same.items() if not ok]

(26, [])

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<span style="display:inline-block;margin-bottom:10px;padding:2px 12px;border-radius:999px;background:#6366f1;color:#fff;font:600 11px/20px system-ui,sans-serif;letter-spacing:.05em;text-transform:uppercase">Done</span>
<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Run it</h2>
<pre><code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">cd course/build
python -m app                                              # development: http://127.0.0.1:8000
APP_CONFIG=config/production.toml OPENAI_API_KEY=... python -m app
docker build -t desk . &amp;&amp; docker run -e OPENAI_API_KEY=... -v deskdata:/data -p 8000:8000 desk</code></pre>
<p style="margin:.55em 0">In production, put a TLS proxy in front (the browser needs https/wss for the microphone) that allows websocket upgrades
on <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">/ws</code>, and point a monitor at <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">/healthz</code> and a scraper at <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">/metrics</code>.</p>
<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">What you built</h2>
<ul style="margin:.4em 0;padding-left:1.4em">
  <li style="margin:.2em 0"><b>Grow it by writing functions:</b> a typed, documented function in <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">app/skills/</code> is discovered, described,
      validated, guarded and timed out; a new toolset is one name in config.</li>
  <li style="margin:.2em 0"><b>Tune it without code:</b> prompts are files, every parameter is config, everything can come from the environment.</li>
  <li style="margin:.2em 0"><b>Trust it:</b> state and guards in code, codes never seen by the model, dead connections recovered, limits, logs,
      metrics, a graceful drain, and a free self-check.</li>
</ul>

</div>